# FST-Arch: run all experiments (GPU)

This notebook reproduces every experiment of the article **Growing and pruning neural networks by fixed-sequence testing with family-wise error control**, on a GPU.

* **Part 1: simulation (random-feature networks).** Oracle risks, the growing and pruning procedures (Tables 1–2), and the exact size of each standard error over $n$ and $R$. All seeds of an architecture are fitted in one batch on the GPU.
* **Part 2: trained networks.** Larger MLPs and ResNets on CIFAR-10, trained on the GPU, analysed with the same tests.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is enough). Then Runtime → Run all.

In [ ]:
import os, sys, json, time, subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', gpu if gpu else 'NOT FOUND -- select a GPU runtime (the code also runs on CPU, more slowly)')
if not os.path.exists('fst-architecture-selection'):
    !git clone -q https://github.com/yamenetoo/fst-architecture-selection.git
%cd fst-architecture-selection/experiments
!git pull -q
EXP_DIR = os.getcwd()
import torch
print('PyTorch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Part 1: simulation (random-feature networks)

## 1.0 Check the GPU backend
The GPU code must reproduce the CPU reference implementation. This takes a few seconds.

In [ ]:
!python check_gpu.py --device {DEVICE}

## 1.1 True architecture risks
$\mu(a)$ for every width $\{4,\dots,256\}$ and depth $\{1,2,3\}$, and for both pruning scenarios (A: width 256, B: width 64 in 5% steps), from 300 seeds and 40 000 test points. Lower `--seed-batch` if the GPU runs out of memory.

In [ ]:
!python oracle.py --device {DEVICE} --seed-batch 25

## 1.2 Choose the margins
$\gamma$ is the smallest improvement worth growing for and $\delta$ the largest loss accepted when pruning, both in squared-error units. The table shows which null hypotheses are true. The width chain should contain **both** true and false nulls, and pruning scenario B should have per-step losses below $\delta$ while the total loss passes it.

In [ ]:
B = 1000        # replications for Tables 1-2 (100 for a quick test)
B_GRID = 500    # replications for the size grid (100 for a quick test)
GAMMA = 0.005   # growing margin
DELTA = 0.01    # pruning margin

import numpy as np
from oracle import WIDTHS, FRACS, FRACS2, PRUNE2_W
orc = np.load('oracle.npz')
mu, mu_pr, mu_pr2 = orc['risk'].mean(0), orc['prune'].mean(0), orc['prune2'].mean(0)
print('Growing, width chain at depth 1: H0 true if mu(w+) - mu(w) >= -gamma')
for i in range(len(WIDTHS) - 1):
    th = mu[i + 1, 0] - mu[i, 0]
    print(f'  {WIDTHS[i]:>3} -> {WIDTHS[i+1]:>3}: theta = {th:+.4f}   H0 {"TRUE" if th >= -GAMMA else "false"}')
print('\nPruning A (width 256): total loss mu(a_k) - mu(a_0)')
for k, fr in enumerate(FRACS[1:], 1):
    print(f'  remove {fr:.0%}: {mu_pr[k] - mu_pr[0]:+.4f}   cumulative H0 {"TRUE" if mu_pr[k] - mu_pr[0] >= DELTA else "false"}')
print(f'\nPruning B (width {PRUNE2_W}): total loss and step loss')
for k, fr in enumerate(FRACS2[1:], 1):
    tot, step = mu_pr2[k] - mu_pr2[0], mu_pr2[k] - mu_pr2[k - 1]
    print(f'  remove {fr:.0%}: total {tot:+.4f} ({"H0 TRUE" if tot >= DELTA else "false"})   step {step:+.4f}')

## 1.3 Growing and pruning procedures (Tables 1 and 2)

In [ ]:
!python sim.py {B} {GAMMA} {DELTA} --device {DEVICE}

In [ ]:
for f in ['table_building.tex', 'table_pruning.tex']:
    print(f'--- {f}'); print(open(f).read())

## 1.4 Exact size over $n$ and $R$
Each width pair is tested at the exact boundary $c=\theta_{\text{true}}$, so a calibrated test rejects in 5% of replications.

In [ ]:
!python grid.py {B_GRID} --device {DEVICE}
print(open('table_grid.tex').read())

## 1.5 Plots

In [ ]:
import matplotlib.pyplot as plt
res, grid = json.load(open('results.json')), json.load(open('grid.json'))
names = ['crossed', 'crossed+safety', 'seed_only', 'single', 'welch']
labels = ['Crossed\n(proposed)', 'Crossed\n+ safety', 'Seed-only', 'Single\nseed', 'Welch']
fig, ax = plt.subplots(1, 3, figsize=(17, 4.3))
ax[0].bar(labels, [res['build'][m]['fwer'] for m in names],
          yerr=[1.96 * res['build'][m]['fwer_se'] for m in names], color='#4c72b0')
ax[0].axhline(0.05, ls='--', color='k', lw=1); ax[0].set_title('Growing: family-wise error rate')
methods, colors = ['crossed', 'seed_only', 'single', 'welch'], ['#4c72b0', '#dd8452', '#c44e52', '#8172b3']
ns = [50, 100, 250, 1000]
for m, c in zip(methods, colors):
    for R, ls in [(5, ':'), (10, '-'), (20, '--')]:
        ax[1].plot(ns, [grid[f'n={n},R={R}'][m] for n in ns], ls=ls, color=c, marker='o', label=m if R == 10 else None)
ax[1].axhline(0.05, color='k', lw=1); ax[1].set_xscale('log'); ax[1].set_xlabel('validation size n')
ax[1].set_title('Exact size (: R=5, - R=10, -- R=20)'); ax[1].legend(fontsize=8)
for key, off, lab in [('prune', -0.2, 'A: width 256'), ('prune2', 0.2, f'B: width {PRUNE2_W}')]:
    ax[2].bar(np.arange(2) + off, [res[key][k]['harm_exceeds'] for k in ['cumulative', 'stepwise']], width=0.4,
              yerr=[1.96 * res[key][k]['harm_exceeds_se'] for k in ['cumulative', 'stepwise']], label=lab)
ax[2].set_xticks([0, 1]); ax[2].set_xticklabels(['Cumulative\n(proposed)', 'Stepwise'])
ax[2].axhline(0.05, ls='--', color='k', lw=1); ax[2].legend()
ax[2].set_title(r'Pruning: $P\{\mu(\hat a)-\mu(a_0)\geq\delta\}$')
plt.tight_layout(); plt.savefig('results.png', dpi=200); plt.show()
print('Exact size at the boundary in the main design (n=250, R=10), mean / max over width pairs:')
for m, v in res['boundary'].items():
    print(f"  {m:<10} {v['mean']:.3f} / {v['max']:.3f}")

# Part 2: trained networks (MLPs and ResNets on CIFAR-10)

Each architecture is trained with the same $M$ seeds on the GPU, and its per-point losses on a held-out pool are stored. The Monte Carlo replications resample seeds and points from that pool (see the README). Set `RUN_PART2 = False` to skip this part.

`QUICK = True` checks that everything works (minutes). `QUICK = False` is the full run (one to two hours on a T4). Set `USE_DRIVE = True` to keep the trained pools in Google Drive: training resumes where it stopped.

In [ ]:
RUN_PART2 = True
QUICK = True
BIG = False          # ResNet widths up to 128 and ResNet-32
USE_DRIVE = False

POOL_ROOT = os.path.join(EXP_DIR, 'deep')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    POOL_ROOT = '/content/drive/MyDrive/fst_pools'
    os.makedirs(POOL_ROOT, exist_ok=True)
if QUICK:
    MLP_ARGS = '--seeds 6 --epochs 40'
    RES_ARGS = '--widths 16 32 64 --depths 1 2 3 --seeds 6 --epochs 10 --prune-width 64 --prune-depth 3'
    B2, B2_GRID = 300, 200
else:
    MLP_ARGS = '--seeds 10 --epochs 60'
    RES_ARGS = '--widths 8 16 32 64 --depths 1 2 3 --seeds 10 --epochs 15 --prune-width 64 --prune-depth 3'
    B2, B2_GRID = 1000, 500
if BIG:
    RES_ARGS = '--widths 16 32 64 128 --depths 1 2 3 5 --seeds 10 --epochs 20 --prune-width 128 --prune-depth 5'
MLP_GAMMA, MLP_DELTA = 0.005, 0.01     # squared-error units
RES_GAMMA, RES_DELTA = 0.005, 0.01     # 0-1 error: 0.5 and 1 percentage points
%cd {EXP_DIR}/deep

In [ ]:
if RUN_PART2:
    t0 = time.time()
    !python train_pool.py --task mlp {MLP_ARGS} --device {DEVICE} --out {POOL_ROOT}/pool_mlp
    print(f'MLP training: {(time.time() - t0) / 60:.1f} min')
    !python analyze_pool.py --task mlp --pool {POOL_ROOT}/pool_mlp --gamma {MLP_GAMMA} --delta {MLP_DELTA} --B {B2} --B-grid {B2_GRID}

On **Kaggle with two GPUs**, train the ResNets with two processes:
```
!python train_pool.py --task resnet {RES_ARGS} --out {POOL_ROOT}/pool_resnet --device cuda:0 --shard 0/2 & \
 python train_pool.py --task resnet {RES_ARGS} --out {POOL_ROOT}/pool_resnet --device cuda:1 --shard 1/2 ; wait
```

In [ ]:
if RUN_PART2:
    t0 = time.time()
    !python train_pool.py --task resnet {RES_ARGS} --device {DEVICE} --out {POOL_ROOT}/pool_resnet
    print(f'ResNet training: {(time.time() - t0) / 60:.1f} min')
    !python analyze_pool.py --task resnet --loss err --pool {POOL_ROOT}/pool_resnet --gamma {RES_GAMMA} --delta {RES_DELTA} --B {B2} --B-grid {B2_GRID}

In [ ]:
for tag in ['mlp_se', 'resnet_err']:
    f = f'results_deep_{tag}.json'
    if not os.path.exists(f):
        continue
    r = json.load(open(f))
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
    ks = list(r['size'])
    ns = sorted({int(k.split(',')[0][2:]) for k in ks}); Rs = sorted({int(k.split(',')[1][2:]) for k in ks})
    for m, c in zip(methods, colors):
        for R, ls in zip(Rs, [':', '-', '--', '-.']):
            ax[0].plot(ns, [r['size'][f'n={n},R={R}'][m]['mean'] for n in ns], ls=ls, color=c, marker='o',
                       label=m if R == Rs[len(Rs) // 2] else None)
    ax[0].axhline(r['alpha'], color='k', lw=1); ax[0].set_xscale('log'); ax[0].set_xlabel('n')
    ax[0].set_title(f'{tag}: exact size (line style = R in {Rs})'); ax[0].legend(fontsize=8)
    g = r['grow']; gn = list(g)
    ax[1].bar(gn, [g[m]['fwer'] for m in gn], yerr=[1.96 * g[m]['fwer_se'] for m in gn], color='#4c72b0')
    ax[1].axhline(r['alpha'], ls='--', color='k', lw=1); ax[1].set_title('growing: FWER'); ax[1].tick_params(axis='x', rotation=30)
    if 'prune' in r:
        p = r['prune']
        ax[2].bar(['cumulative', 'stepwise'], [p[k]['harm_exceeds'] for k in ['cumulative', 'stepwise']],
                  yerr=[1.96 * p[k]['harm_exceeds_se'] for k in ['cumulative', 'stepwise']], color=['#4c72b0', '#c44e52'])
        ax[2].axhline(r['alpha'], ls='--', color='k', lw=1); ax[2].set_title(r'pruning: $P\{\mu(\hat a)-\mu(a_0)\geq\delta\}$')
    plt.tight_layout(); plt.savefig(f'deep_{tag}.png', dpi=200); plt.show()

# Download the results

In [ ]:
%cd {EXP_DIR}
out = ['results.json', 'grid.json', 'table_building.tex', 'table_pruning.tex', 'table_grid.tex', 'results.png']
out += ['deep/' + f for f in os.listdir('deep') if f.startswith(('results_deep_', 'table_deep_', 'deep_')) and not f.endswith('.py')]
out = [f for f in out if os.path.exists(f)]
print(out)
try:
    from google.colab import files
    for f in out:
        files.download(f)
except ImportError:
    print('Not on Colab: the files are in', os.getcwd())